# BTW 2025: Prepare vote entries

This notebook converts `btw25_wbz_ergebnisse.csv` and `btw25_rws_bst2.csv` into the shared `VoteEntry` JSON structure.

The published category `m|d|o` is stored under the established internal value `gender="m"` while retaining its broader source meaning. The 2025 birth-year cohorts are represented as `18-24`, `25-34`, `35-44`, `45-59`, `60-69`, and `70+`; they are not forced into the different 2021 boundaries.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "scripts").is_dir() and (candidate / "package.json").is_file():
            return candidate
    raise RuntimeError("Run this notebook inside the repository.")

ROOT = find_repository_root()
sys.path.insert(0, str(ROOT))
DISTRICT_RESULTS_CSV = ROOT / "scripts/data/btw25_wbz_ergebnisse.csv"
STATE_DEMOGRAPHICS_CSV = ROOT / "scripts/data/btw25_rws_bst2.csv"
OUTPUT_DIRECTORY = ROOT / "scripts/data/generated/btw2025"

from scripts.election_data.btw2025 import (
    BTW2025_AGE_GROUPS,
    normalize_state_statistic_rows,
    read_polling_district_csv,
    read_representative_statistics_csv,
    reshape_polling_district_votes,
)
from scripts.election_data.btw2025_statistics import reshape_state_statistic_votes
from scripts.election_data.notebook_steps import (
    aggregate_to_constituencies,
    calculate_demographic_profiles,
    inspect_district_rows,
    normalize_district_rows,
    select_state_statistic_detail_rows,
    select_usable_district_rows,
)
from scripts.election_data.pipeline import distribute_district_votes, write_vote_entries
from scripts.election_data.profiles import build_state_method_profiles
from scripts.election_data.validation import entries_to_frame, validate_vote_entries


In [ ]:
raw_districts = read_polling_district_csv(DISTRICT_RESULTS_CSV)
diagnostics = inspect_district_rows(raw_districts)
display(diagnostics["status"].value_counts(dropna=False))
usable_districts = select_usable_district_rows(raw_districts, diagnostics)
normalized_districts = normalize_district_rows(usable_districts)

first_district_totals = aggregate_to_constituencies(
    reshape_polling_district_votes(normalized_districts, vote_type="1")
)
second_district_totals = aggregate_to_constituencies(
    reshape_polling_district_votes(normalized_districts, vote_type="2")
)
district_totals = pd.concat(
    [first_district_totals, second_district_totals],
    ignore_index=True,
)

raw_statistics = read_representative_statistics_csv(STATE_DEMOGRAPHICS_CSV)
normalized_statistics = normalize_state_statistic_rows(raw_statistics)
detail_statistics = select_state_statistic_detail_rows(normalized_statistics)
statistic_votes = reshape_state_statistic_votes(detail_statistics)
demographic_profiles = calculate_demographic_profiles(statistic_votes)

profiles = build_state_method_profiles(
    district_totals,
    demographic_profiles,
    age_groups=BTW2025_AGE_GROUPS,
)
entries = distribute_district_votes(district_totals, profiles)
first_entries = [entry for entry in entries if entry.voteType == "1"]
second_entries = [entry for entry in entries if entry.voteType == "2"]

first_report = validate_vote_entries(
    first_entries,
    first_district_totals,
    profiles[profiles["voteType"] == "1"],
)
second_report = validate_vote_entries(
    second_entries,
    second_district_totals,
    profiles[profiles["voteType"] == "2"],
)
display(first_report)
display(second_report)

first_path = write_vote_entries(first_entries, OUTPUT_DIRECTORY / "first_votes.json")
second_path = write_vote_entries(second_entries, OUTPUT_DIRECTORY / "second_votes.json")
print(first_path)
print(second_path)

entry_frame = entries_to_frame(entries)
display(
    entry_frame.groupby(["voteType", "party"], as_index=False)["votes"]
    .sum()
    .sort_values(["voteType", "votes"], ascending=[True, False])
    .groupby("voteType")
    .head(15)
)


Run `02_validate_btw2025_vote_entries.ipynb` after generation. The record fields match 2021, but the frontend still needs election-specific age-filter options before these files can be loaded as live application data.
